# Code Generator

Uses an LLM to generate high-performance C++ code from Python.

**Setup:** Add `OPENAI_API_KEY` or `OPENROUTER_API_KEY` to `.env`

In [ ]:
%pip install -q openai gradio python-dotenv

In [ ]:
import os
import re
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

load_dotenv(".env", override=True)
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "").strip()
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY", "").strip()

In [ ]:
if OPENROUTER_API_KEY:
    client = OpenAI(api_key=OPENROUTER_API_KEY, base_url="https://openrouter.ai/api/v1")
    MODEL = "openai/gpt-4o-mini"
    print("Using OpenRouter")
elif OPENAI_API_KEY:
    client = OpenAI(api_key=OPENAI_API_KEY)
    MODEL = "gpt-4o-mini"
    print("Using OpenAI")
else:
    client = None
    MODEL = None
    print("Add OPENAI_API_KEY or OPENROUTER_API_KEY to .env")

In [ ]:
SYSTEM_PROMPT = """You are an expert C++ developer. Convert the given Python code to high-performance C++.
Output only the C++ code. No markdown fences, no explanations."""

def python_to_cpp(python_code):
    if not client:
        return "Error: No API key. Add OPENAI_API_KEY or OPENROUTER_API_KEY to .env"
    if not python_code or not python_code.strip():
        return "Paste Python code above."
    try:
        r = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": python_code}
            ],
            temperature=0.3
        )
        out = (r.choices[0].message.content or "").strip()
        if out.startswith("```"):
            out = re.sub(r"^```\w*\n?", "", out)
            out = re.sub(r"\n?```\s*$", "", out)
        return out.strip()
    except Exception as e:
        return f"Error: {e}"

In [ ]:
with gr.Blocks(title="Code Generator") as demo:
    gr.Markdown("# Python to C++ Code Generator")
    with gr.Row():
        code_in = gr.Code(label="Python", language="python", lines=15)
        code_out = gr.Code(label="C++", language="cpp", lines=15)
    btn = gr.Button("Convert", variant="primary")
    btn.click(fn=python_to_cpp, inputs=code_in, outputs=code_out)

demo.launch()